In [ ]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# ===================================================================
# AUTOR: Carmen Gómez Alarcón | FECHA: 2026
# FASE: Paso 1. Gráficos del sensor interior y variables exteriores
# DESCRIPCIÓN:
#   Carga datos del sensor interior (pasillo), ocupación y variables
#   exteriores (Tª, humedad, radiación solar). Genera scatter plots
#   por mes (dic-mar) correlacionando Tª, CO2, humedad y ocupación,
#   incluyendo relaciones interior-exterior.
#
# UBICACIÓN DEL SENSOR EN EL AULA 1.5:
#   - Corridor (Pasillo): Único sensor, junto a la pared, orientado al pasillo.
# ===================================================================

import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta, time

# Directorio
OUTPUT_DIR = 'plots_scatters_1.5'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# CARGAR DATOS DE OCUPACIÓN (archivo separado)
# ==========================================
print("\n" + "="*60)
print("LOADING OCCUPANCY DATA...")
print("="*60)

OCCUPANCY_FILE = '15 occupancy.xlsx'

if os.path.exists(OCCUPANCY_FILE):
    df_occ = pd.read_excel(OCCUPANCY_FILE)
    print(f"   Loaded: {len(df_occ)} rows from {OCCUPANCY_FILE}")
    
    # Buscar columna de fecha
    date_col = None
    for col in df_occ.columns:
        if 'fecha+hora' in str(col).lower() or 'fecha' in str(col).lower() or 'date' in str(col).lower():
            date_col = col
            break
    if date_col is None:
        date_col = df_occ.columns[0]
    
    df_occ['DateTime'] = pd.to_datetime(df_occ[date_col], errors='coerce')
    df_occ = df_occ.dropna(subset=['DateTime'])
    
    # Buscar columna de ocupación
    occ_col = None
    for col in df_occ.columns:
        if 'occupancy' in str(col).lower() or 'ocupacion' in str(col).lower() or 'personas' in str(col).lower():
            occ_col = col
            break
    
    if occ_col:
        df_occ['Occupancy'] = pd.to_numeric(df_occ[occ_col], errors='coerce')
        df_occ = df_occ.dropna(subset=['Occupancy'])
        df_occ = df_occ[['DateTime', 'Occupancy']].set_index('DateTime')
        print(f"   Occupancy records: {len(df_occ)}")
        print(f"   Date range: {df_occ.index.min()} → {df_occ.index.max()}")
        print(f"   Values: {sorted(df_occ['Occupancy'].unique())}")
    else:
        print("   ❌ No occupancy column found!")
        df_occ = None
else:
    print(f"   ❌ File not found: {OCCUPANCY_FILE}")
    df_occ = None



LOADING OCCUPANCY DATA...
   Loaded: 155 rows from 1.5 occupancy.xlsx
   Occupancy records: 153
   Date range: 2026-02-09 00:00:00 → 2026-03-19 00:00:00
   Values: [np.float64(0.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(12.0), np.float64(13.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0), np.float64(31.0), np.float64(32.0), np.float64(33.0), np.float64(34.0), np.float64(35.0), np.float64(36.0), np.float64(37.0), np.float64(38.0), np.float64(40.0), np.float64(46.0), np.float64(47.0), np.float64(49.0), np.float64(50.0), np.float64(52.0), np.float64(53.0), np.float64(54.0), np.float64(56.0), np.float64(57.0), np.float64(63.0), np.float64(64.0)]


In [ ]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# ===================================================================
# AUTOR: Carmen Gómez Alarcón | FECHA: 2026
# FASE: Paso 1. Gráficos del sensor interior y variables exteriores
# DESCRIPCIÓN:
#   Carga datos del sensor interior (pasillo), ocupación y variables
#   exteriores (Tª, humedad, radiación solar). Genera scatter plots
#   por mes (dic-mar) correlacionando Tª, CO2, humedad y ocupación,
#   incluyendo relaciones interior-exterior.
#
# UBICACIÓN DEL SENSOR EN EL AULA 1.5:
#   - Corridor (Pasillo): Único sensor, junto a la pared, orientado al pasillo.
# ===================================================================

import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import numpy as np

# Directorio
OUTPUT_DIR = 'plots_scatters_1.5'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# CARGAR DATOS DE OCUPACIÓN (archivo separado)
# ==========================================
print("\n" + "="*60)
print("LOADING OCCUPANCY DATA...")
print("="*60)

OCCUPANCY_FILE = '15 occupancy.xlsx'
df_occ = None

if os.path.exists(OCCUPANCY_FILE):
    df_occ_raw = pd.read_excel(OCCUPANCY_FILE)
    print(f"   Loaded: {len(df_occ_raw)} rows")
    print(f"   Columns: {df_occ_raw.columns.tolist()}")
    
    # 🔧 CORRECCIÓN: Buscar columna 'Date' PRIMERO (tiene fecha+hora)
    date_col = None
    # 1º: Buscar columna exacta 'Date'
    for col in df_occ_raw.columns:
        if col == 'Date':
            date_col = col
            break
    # 2º: Si no, buscar 'DateTime' o similar
    if date_col is None:
        for col in df_occ_raw.columns:
            if col in ['DateTime', 'Datetime']:
                date_col = col
                break
    # 3º: Si no, buscar por tipo datetime64 con horas
    if date_col is None:
        for col in df_occ_raw.columns:
            if df_occ_raw[col].dtype == 'datetime64[ns]':
                if df_occ_raw[col].dt.hour.nunique() > 1:
                    date_col = col
                    break
    # 4º: Si no, buscar por nombre
    if date_col is None:
        for col in df_occ_raw.columns:
            col_lower = str(col).lower()
            if any(x in col_lower for x in ['date', 'fecha', 'hora', 'time', 'datetime']):
                date_col = col
                break
    # 5º: Último recurso
    if date_col is None:
        date_col = df_occ_raw.columns[0]
    
    print(f"   Date column: '{date_col}'")
    
    # Convertir a datetime
    df_occ_raw['DateTime'] = pd.to_datetime(df_occ_raw[date_col], errors='coerce', dayfirst=True)
    df_occ_raw = df_occ_raw.dropna(subset=['DateTime'])
    print(f"   Valid dates: {len(df_occ_raw)}")
    print(f"   Date range: {df_occ_raw['DateTime'].min()} → {df_occ_raw['DateTime'].max()}")
    
    # Buscar columna de ocupación
    occ_col = None
    for col in df_occ_raw.columns:
        col_lower = str(col).lower()
        if any(x in col_lower for x in ['occup', 'person', 'ocup', 'people', 'nº']):
            occ_col = col
            break
    if occ_col is None:
        for col in df_occ_raw.columns:
            if col != date_col and col != 'DateTime':
                if df_occ_raw[col].dtype in ['int64', 'float64']:
                    if 'li' not in str(col).lower() and 'ld' not in str(col).lower() and 'unnamed' not in str(col).lower():
                        occ_col = col
                        break
    if occ_col is None:
        occ_col = df_occ_raw.columns[-1]
    
    print(f"   Occupancy column: '{occ_col}'")
    
    # Crear serie de ocupación indexada por fecha
    df_occ_raw['Occupancy'] = pd.to_numeric(df_occ_raw[occ_col], errors='coerce')
    df_occ = df_occ_raw.dropna(subset=['Occupancy'])[['DateTime', 'Occupancy']].copy()
    df_occ = df_occ.set_index('DateTime').sort_index()
    print(f"   Valid occupancy records: {len(df_occ)}")
    print(f"   Occupancy values: {sorted(df_occ['Occupancy'].unique())[:20]}...")
    print(f"   Max occupancy: {df_occ['Occupancy'].max()}")
else:
    print(f"   ⚠️ File not found: {OCCUPANCY_FILE}")

# ==========================================
# CARGAR DATOS DE SENSORES
# ==========================================
file_pattern = '1.5_data_*_week_*.xlsx'
all_files = sorted(glob.glob(file_pattern))

if not all_files:
    print(f"\nERROR: No files found matching '{file_pattern}'")
    exit()

print(f"\nFound {len(all_files)} weekly files")
print("\n" + "="*60)
print("LOADING SENSOR DATA...")
print("="*60)

all_records = []

for file_path in all_files:
    fname = os.path.basename(file_path)
    print(f"   {fname}...", end=" ")

    df = pd.read_excel(file_path)
    metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']
    date_cols = [c for c in df.columns if c not in metadata_cols]
    timestamps = pd.to_datetime(date_cols, errors='coerce')

    def get_series(df, variable, location):
        mask = (df['Variable_Measure'] == variable) & (df['Sensor_Location'] == location)
        row = df[mask]
        if len(row) > 0:
            return row[date_cols].iloc[0].astype(float)
        return pd.Series(np.nan, index=date_cols)

    # Sensor Corridor
    temp_corridor = get_series(df, 'Temperature', 'Corridor')
    co2_corridor  = get_series(df, 'CO2', 'Corridor')
    hum_corridor  = get_series(df, 'Humidity', 'Corridor')

    # Outdoor
    temp_outdoor  = get_series(df, 'Temperature_Outdoor', 'Outdoor')
    hum_outdoor   = get_series(df, 'Humidity_Outdoor', 'Outdoor')
    solar_outdoor = get_series(df, 'Solar_Radiation', 'Outdoor')

    # DataFrame base
    df_mini = pd.DataFrame({
        'Temperature_Corridor': temp_corridor.values,
        'CO2_Corridor':         co2_corridor.values,
        'Humidity_Corridor':    hum_corridor.values,
        'Temperature_Outdoor':  temp_outdoor.values,
        'Humidity_Outdoor':     hum_outdoor.values,
        'Solar_Radiation':      solar_outdoor.values,
    }, index=timestamps)

    # Añadir ocupación desde archivo externo
    occ_count = 0
    if df_occ is not None:
        occupancy_values = []
        for ts in timestamps:
            if pd.isnull(ts):
                occupancy_values.append(np.nan)
                continue
            time_diffs = abs(df_occ.index - ts)
            min_diff = time_diffs.min()
            if min_diff <= pd.Timedelta('30min'):
                closest_idx = time_diffs.argmin()
                occupancy_values.append(df_occ.iloc[closest_idx]['Occupancy'])
            else:
                occupancy_values.append(np.nan)
        df_mini['Occupancy'] = occupancy_values
        occ_count = sum(1 for v in occupancy_values if not pd.isna(v))
    
    all_records.append(df_mini)
    print(f"OK ({len(df_mini)} records, {occ_count} with occupancy)")

# Concatenar todos los datos
print("\n   Concatenating all data...")
df_all = pd.concat(all_records, ignore_index=False)
df_all = df_all.sort_index()
df_all['Month'] = df_all.index.month_name()

print(f"   Total records: {len(df_all)}")
print(f"   Months: {df_all['Month'].unique().tolist()}")

if 'Occupancy' in df_all.columns:
    occ_valid = df_all['Occupancy'].notna().sum()
    print(f"   Records with occupancy: {occ_valid} / {len(df_all)}")
    if occ_valid > 0:
        print(f"   Occupancy values: {sorted(df_all['Occupancy'].dropna().unique())}")
        print(f"   Occupancy range: {df_all['Occupancy'].min():.0f} - {df_all['Occupancy'].max():.0f}")

# ==========================================
# CONFIGURACIÓN
# ==========================================
month_order = ['December', 'January', 'February', 'March']
unique_months = [m for m in month_order if m in df_all['Month'].unique()]
print(f"\n   Months with data: {unique_months}")

sensor_colors = {'Corridor': '#2196F3'}
sensor_markers = {'Corridor': 'o'}

# ==========================================
# FUNCIÓN: Scatter interior (1 sensor)
# ==========================================
def create_scatter_1sensor(x_col, y_col, x_label, y_label, suptitle, filename):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(suptitle, fontsize=15, fontweight='bold', y=1.01)

    axes_flat = axes.flatten()
    month_positions = {'December': 0, 'January': 1, 'February': 2, 'March': 3}
    any_data = False

    for month, pos in month_positions.items():
        ax = axes_flat[pos]

        if month not in unique_months:
            ax.set_visible(False)
            continue

        data_month = df_all[df_all['Month'] == month]

        if x_col not in data_month.columns or y_col not in data_month.columns:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        df_loc = data_month.dropna(subset=[x_col, y_col])

        if len(df_loc) < 3:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, f'Only {len(df_loc)} points', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        any_data = True

        ax.scatter(df_loc[x_col], df_loc[y_col],
                   c=sensor_colors['Corridor'],
                   marker=sensor_markers['Corridor'],
                   alpha=0.5, s=15, edgecolors='none',
                   label='Corridor')

        # Línea de tendencia
        try:
            x_v = df_loc[x_col].values
            y_v = df_loc[y_col].values
            if len(x_v) > 2 and len(y_v) > 2:
                z = np.polyfit(x_v, y_v, 1)
                p = np.poly1d(z)
                x_t = np.linspace(x_v.min(), x_v.max(), 100)
                ax.plot(x_t, p(x_t), color='red', linewidth=1.5,
                       linestyle='--', alpha=0.8)
                r2 = np.corrcoef(x_v, y_v)[0, 1] ** 2
                ax.text(0.05, 0.95, f'R² = {r2:.3f}', 
                       transform=ax.transAxes, fontsize=9,
                       verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        except Exception:
            pass

        # 🔧 Ajustar límites para que no se corte el eje
        x_min = df_loc[x_col].min()
        x_max = df_loc[x_col].max()
        x_margin = (x_max - x_min) * 0.1 if x_max > x_min else 5
        ax.set_xlim(x_min - x_margin, x_max + x_margin)
        
        y_min = df_loc[y_col].min()
        y_max = df_loc[y_col].max()
        y_margin = (y_max - y_min) * 0.1 if y_max > y_min else 10
        ax.set_ylim(y_min - y_margin, y_max + y_margin)

        ax.set_title(f'{month}  |  n={len(df_loc)}', fontsize=11, fontweight='bold')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)
        ax.legend(loc='upper right', fontsize=9, framealpha=0.9)

    if not any_data:
        print(f"   SKIPPED: No data for {suptitle}")
        plt.close()
        return

    plt.tight_layout()
    file_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(file_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")

# ==========================================
# FUNCIÓN: Scatter outdoor vs interior (1 sensor)
# ==========================================
def create_scatter_outdoor_1sensor(x_var, y_col, x_label, y_label, suptitle, filename):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(suptitle, fontsize=15, fontweight='bold', y=1.01)

    axes_flat = axes.flatten()
    month_positions = {'December': 0, 'January': 1, 'February': 2, 'March': 3}
    any_data = False

    for month, pos in month_positions.items():
        ax = axes_flat[pos]

        if month not in unique_months:
            ax.set_visible(False)
            continue

        data_month = df_all[df_all['Month'] == month]

        if x_var not in data_month.columns or y_col not in data_month.columns:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        df_loc = data_month.dropna(subset=[x_var, y_col])

        if len(df_loc) < 3:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, f'Only {len(df_loc)} points', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        any_data = True

        ax.scatter(df_loc[x_var], df_loc[y_col],
                   c=sensor_colors['Corridor'],
                   marker=sensor_markers['Corridor'],
                   alpha=0.5, s=15, edgecolors='none',
                   label='Corridor')

        try:
            x_v = df_loc[x_var].values
            y_v = df_loc[y_col].values
            if len(x_v) > 2 and len(y_v) > 2:
                z = np.polyfit(x_v, y_v, 1)
                p = np.poly1d(z)
                x_t = np.linspace(x_v.min(), x_v.max(), 100)
                ax.plot(x_t, p(x_t), color='red', linewidth=1.5,
                       linestyle='--', alpha=0.8)
                r2 = np.corrcoef(x_v, y_v)[0, 1] ** 2
                ax.text(0.05, 0.95, f'R² = {r2:.3f}', 
                       transform=ax.transAxes, fontsize=9,
                       verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        except Exception:
            pass

        # 🔧 Ajustar límites
        x_min = df_loc[x_var].min()
        x_max = df_loc[x_var].max()
        x_margin = (x_max - x_min) * 0.1 if x_max > x_min else 5
        ax.set_xlim(x_min - x_margin, x_max + x_margin)
        
        y_min = df_loc[y_col].min()
        y_max = df_loc[y_col].max()
        y_margin = (y_max - y_min) * 0.1 if y_max > y_min else 10
        ax.set_ylim(y_min - y_margin, y_max + y_margin)

        ax.set_title(f'{month}  |  n={len(df_loc)}', fontsize=11, fontweight='bold')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)
        ax.legend(loc='upper right', fontsize=9, framealpha=0.9)

    if not any_data:
        print(f"   SKIPPED: No data for {suptitle}")
        plt.close()
        return

    plt.tight_layout()
    file_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(file_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")

# ==========================================
# GENERAR GRÁFICOS
# ==========================================
print("\n" + "="*60)
print("GENERATING SCATTER PLOTS (Corridor sensor)")
print("="*60)

print("\n--- INTERIOR VARIABLES ---")

create_scatter_1sensor('Temperature_Corridor', 'CO2_Corridor',
    'Temperature (°C)', 'CO₂ (ppm)',
    'CO₂ vs Temperature — Classroom 1.5',
    '01_CO2_vs_Temperature.png')

create_scatter_1sensor('CO2_Corridor', 'Temperature_Corridor',
    'CO₂ (ppm)', 'Temperature (°C)',
    'Temperature vs CO₂ — Classroom 1.5',
    '02_Temperature_vs_CO2.png')

create_scatter_1sensor('CO2_Corridor', 'Humidity_Corridor',
    'CO₂ (ppm)', 'Humidity (%)',
    'Humidity vs CO₂ — Classroom 1.5',
    '03_Humidity_vs_CO2.png')

create_scatter_1sensor('Humidity_Corridor', 'CO2_Corridor',
    'Humidity (%)', 'CO₂ (ppm)',
    'CO₂ vs Humidity — Classroom 1.5',
    '04_CO2_vs_Humidity.png')

create_scatter_1sensor('Humidity_Corridor', 'Temperature_Corridor',
    'Humidity (%)', 'Temperature (°C)',
    'Temperature vs Humidity — Classroom 1.5',
    '05_Temperature_vs_Humidity.png')

# CO₂ vs Occupancy
if 'Occupancy' in df_all.columns and df_all['Occupancy'].notna().sum() > 3:
    create_scatter_1sensor('Occupancy', 'CO2_Corridor',
        'Occupancy (persons)', 'CO₂ (ppm)',
        'CO₂ vs Occupancy — Classroom 1.5',
        '06_CO2_vs_Occupancy.png')
else:
    occ_count = df_all['Occupancy'].notna().sum() if 'Occupancy' in df_all.columns else 0
    print(f"   SKIPPED CO₂ vs Occupancy: only {occ_count} occupancy records")

print("\n--- OUTDOOR VARIABLES ---")

create_scatter_outdoor_1sensor('Temperature_Outdoor', 'Temperature_Corridor',
    'Outdoor Temperature (°C)', 'Indoor Temperature (°C)',
    'Indoor vs Outdoor Temperature — Classroom 1.5',
    '11_Indoor_vs_Outdoor_Temperature.png')

create_scatter_outdoor_1sensor('Humidity_Outdoor', 'Humidity_Corridor',
    'Outdoor Humidity (%)', 'Indoor Humidity (%)',
    'Indoor vs Outdoor Humidity — Classroom 1.5',
    '12_Indoor_vs_Outdoor_Humidity.png')

create_scatter_outdoor_1sensor('Solar_Radiation', 'Temperature_Corridor',
    'Solar Radiation (W/m²)', 'Indoor Temperature (°C)',
    'Indoor Temperature vs Solar Radiation — Classroom 1.5',
    '13_Temperature_vs_Solar_Radiation.png')

create_scatter_outdoor_1sensor('Temperature_Outdoor', 'CO2_Corridor',
    'Outdoor Temperature (°C)', 'Indoor CO₂ (ppm)',
    'Indoor CO₂ vs Outdoor Temperature — Classroom 1.5',
    '14_CO2_vs_Outdoor_Temperature.png')

print("\n" + "="*60)
print("AULA 1.5 PLOTS COMPLETE!")
print("="*60)
print(f"\nFiles saved in '{OUTPUT_DIR}/'")


LOADING OCCUPANCY DATA...
   Loaded: 155 rows
   Columns: ['Unnamed: 0', 'Fecha', 'Hora', 'Date', 'LI (pasillo)', 'LD (jardín)', 'Occupancy']
   Date column: 'Date'
   Valid dates: 155
   Date range: 2026-02-09 09:50:00 → 2026-03-29 04:33:36
   Occupancy column: 'Occupancy'
   Valid occupancy records: 153
   Occupancy values: [np.float64(0.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(12.0), np.float64(13.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0)]...
   Max occupancy: 64.0

Found 12 weekly files

LOADING SENSOR DATA...
   1.5_data_December_2025_week_4_days_22-31.xlsx... OK (849 records, 0 with occupancy)
   1.5_data_February_2026_week_1_days_1-7.xlsx... OK (530 records, 0 with occupancy)
   1.5_data_February_2026_week_2_days_8-14.xlsx... OK (566 records, 181 